<a href="https://colab.research.google.com/github/ALIAB1054/assign-1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — Data Contract
## Lane 2: Refresh / Content Opportunity Scoring

This notebook proves the data contract for my lane, verifies three facts about the slice with real queries, builds five features, and demonstrates the leakage trap on real warehouse data.

## Setup

Connect to the Hugging Face-hosted warehouse release via DuckDB. Requires an `HF_TOKEN` Colab Secret (plain Read token, gated-repositories access accepted in browser first).

In [1]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_query_90d            2,414,248 rows


## 1)–2) The Contract

**Lane 2: Refresh / Content Opportunity Scoring**

1. **Unit of analysis:** One row = one content item (`content_hash_id`) for one client, with features aggregated over a defined prior window.

2. **Table(s) used:** `fact_content_daily_performance` (daily GSC performance, filtered to a mid-panel period for feature development), joined with `dim_content` (content metadata) and `dim_clients` (client-level grouping and history-depth checks).

3. **Time window:** Features from a prior 90-day window ending mid-March 2026; label from the following 30-day window (through mid-April 2026). This is a genuine past → future split, not a same-window bucket — developed on a mid-panel period, never on the sealed final month (June 2026 / `_sample`).

4. **Label/proxy:** `is_declining` — impressions in the 30-day target window fall more than 20% versus the 30-day period immediately preceding it inside the prior feature window. This is a proxy for "this page needs review," not a guarantee that refreshing it would cause recovery.

5. **Deliberate exclusion:** Any FlyRank product decision output — `health_score`, `priority_score`, `action_type`, `needs_ctr_fix`, or similar refresh-tier flags. These are the product's own decisions, not observable signals, and using them as features would leak the answer rather than discover it. (They are also not shipped in this dataset by design.)

## 3) Three Verification Queries

### Query 1 — Grain check
Prove one row really is one (client, content, date) combination.

In [4]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-01-01' AND report_date < '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()

print(f"Duplicate (client, content, date) rows: {len(grain_check)}")  # expect 0
grain_check.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, date) rows: 0


,client_hash_id,content_hash_id,report_date,row_count


### Query 2 — Slice size
Row count and date span for the working window.

In [5]:
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS earliest_date,
           MAX(report_date) AS latest_date,
           COUNT(DISTINCT content_hash_id) AS unique_content_items,
           COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-01-01' AND report_date < '2026-04-01'
""").df()

slice_stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,earliest_date,latest_date,unique_content_items,unique_clients
0,25087303,2026-01-01,2026-03-31,349411,59


### Query 3 — Availability
First check the real schema to find the correct boolean/flag column, then filter with `IS TRUE`.

In [9]:
con.sql(f"CREATE OR REPLACE TEMPORARY VIEW tmp_fact_daily AS SELECT * FROM {TABLES['fact_daily']} LIMIT 0;")
con.sql("PRAGMA table_info('tmp_fact_daily')").df()

,cid,name,type,notnull,dflt_value,pk
0,0,report_date,DATE,False,None,False
1,1,client_hash_id,VARCHAR,False,None,False
2,2,content_hash_id,VARCHAR,False,None,False
3,3,client_has_gsc,BOOLEAN,False,None,False
4,4,client_has_ga4,BOOLEAN,False,None,False
5,5,gsc_data_available,BOOLEAN,False,None,False
6,6,ga4_data_available,BOOLEAN,False,None,False
7,7,gsc_impressions,BIGINT,False,None,False
8,8,gsc_clicks,BIGINT,False,None,False
9,9,gsc_sum_position,BIGINT,False,None,False


In [10]:
# Update the column name below once confirmed from the DESCRIBE output above
# (ga4_data_available is the likely candidate per the lane guide's panel-coverage note)

total_rows = con.sql(f"""
    SELECT COUNT(*) FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-01-01' AND report_date < '2026-04-01'
""").fetchone()[0]

available_rows = con.sql(f"""
    SELECT COUNT(*) FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-01-01' AND report_date < '2026-04-01'
      AND ga4_data_available IS TRUE
""").fetchone()[0]

print(f"Total rows: {total_rows:,}")
print(f"GA4-available rows: {available_rows:,}")
print(f"Survival rate: {available_rows/total_rows:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 25,087,303
GA4-available rows: 675,073
Survival rate: 2.7%


## Five Features

Built from the prior 90-day window (2025-12-15 through 2026-03-15), each with an "available when" justification.

In [11]:
features = con.sql(f"""
    WITH prior_window AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2025-12-15' AND report_date < '2026-03-15'
    )
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS total_impressions,
           SUM(gsc_clicks) AS total_clicks,
           AVG(gsc_avg_position) AS avg_position,
           STDDEV(gsc_avg_position) AS position_volatility,
           COUNT(DISTINCT report_date) AS days_with_data
    FROM prior_window
    GROUP BY 1, 2
    HAVING total_impressions >= 100
""").df()

print(f"{len(features):,} content items with enough history")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

106,697 content items with enough history


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,position_volatility,days_with_data
0,client_3ffa76342f366962,content_4c65659252a4d124,159.0,7.0,6.715513,10.503760,88
1,client_3ffa76342f366962,content_14533db2b07dd42a,255.0,3.0,4.864123,2.877842,86
2,client_9958f0a7ae1df715,content_a9ae30aa11e6eef0,134.0,0.0,45.368275,20.814388,90
3,client_9958f0a7ae1df715,content_c87de72c9b9aac06,207.0,0.0,22.257112,14.892292,90
4,client_9958f0a7ae1df715,content_d87acbcda869b801,587.0,2.0,12.196018,10.711812,90


**Feature notes — "knowable at the decision moment because…"**

1. `total_impressions` — sum of impressions from the 90-day prior window, entirely historical relative to the decision point.
2. `total_clicks` — same: aggregated from observed past clicks only.
3. `avg_position` — average of realized rankings already observed in the prior window, not a forward estimate.
4. `position_volatility` — standard deviation of the same historical position series; describes past instability, not future movement.
5. `days_with_data` — count of reporting days observed in the prior window, known as soon as that window closes.

## 4) The Leakage Trap

Build a genuine future-window label, add one label-derived column on purpose, watch the score jump toward perfect, then delete it and keep the honest number.

In [12]:
labels = con.sql(f"""
    WITH target_window AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_target_30d
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-15' AND report_date < '2026-04-14'
        GROUP BY 1
    ),
    prior_last30 AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_prior_last30d
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-02-13' AND report_date < '2026-03-15'
        GROUP BY 1
    )
    SELECT t.content_hash_id,
           CASE WHEN t.imp_target_30d < 0.8 * p.imp_prior_last30d THEN 1 ELSE 0 END AS is_declining,
           (t.imp_target_30d - p.imp_prior_last30d) * 1.0 / NULLIF(p.imp_prior_last30d, 0) AS pct_change
    FROM target_window t
    JOIN prior_last30 p ON t.content_hash_id = p.content_hash_id
""").df()

data = features.merge(labels, on='content_hash_id', how='inner')
print(f"{len(data):,} rows after merging features with labels")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,659 rows after merging features with labels


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,position_volatility,days_with_data,is_declining,pct_change
0,client_3ffa76342f366962,content_4c65659252a4d124,159.0,7.0,6.715513,10.503760,88,1,-0.701299
1,client_3ffa76342f366962,content_14533db2b07dd42a,255.0,3.0,4.864123,2.877842,86,1,-0.339623
2,client_9958f0a7ae1df715,content_a9ae30aa11e6eef0,134.0,0.0,45.368275,20.814388,90,0,0.052632
3,client_9958f0a7ae1df715,content_c87de72c9b9aac06,207.0,0.0,22.257112,14.892292,90,1,-0.397260
4,client_9958f0a7ae1df715,content_d87acbcda869b801,587.0,2.0,12.196018,10.711812,90,1,-0.483516


In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

feature_cols = ['total_impressions', 'total_clicks', 'avg_position', 'position_volatility', 'days_with_data']
X_honest = data[feature_cols].fillna(0)
y = data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
honest_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
honest_score = precision_score(y_te, honest_model.predict(X_te))

print(f"Honest precision: {honest_score:.3f}")

Honest precision: 0.000


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
# The trap: add the label-derived column on purpose
X_leaky = X_honest.copy()
X_leaky['pct_change'] = data['pct_change'].fillna(0)  # this IS the leak — computed from the target window itself

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr_l, y_tr_l)
leaky_score = precision_score(y_te_l, leaky_model.predict(X_te_l))

print(f"Leaky precision (pct_change included): {leaky_score:.3f}  <- artificially inflated")

Leaky precision (pct_change included): 1.000  <- artificially inflated


In [15]:
# Delete the leak, keep the honest number
print(f"Final honest precision (leak removed): {honest_score:.3f}")
print(f"Leaky precision was: {leaky_score:.3f}")
print(f"Inflation from the leak: {leaky_score - honest_score:+.3f}")

Final honest precision (leak removed): 0.000
Leaky precision was: 1.000
Inflation from the leak: +1.000


## 4 (continued) — Limitation

This slice uses a prior-90-day / target-30-day split centered on Q1 2026. Per the lane guide's unbalanced-panel note, only 4 of 70 clients have 12+ months of history (confirmed in notebook 03's panel check) — clients with shorter history may be systematically under-represented in this window, and the feature distributions here may not generalize to newer clients or to clients not yet using GA4 (`ga4_data_available = FALSE`).

## 5) Self-check

- [x] 5 contract answers, naming Lane 2 (Refresh / Content Opportunity Scoring) explicitly
- [x] 3 verification queries run, outputs visible, availability checked with `IS TRUE`
- [x] 5 features, each with an "available when" justification line
- [x] Genuine past → future label (prior 90d → target 30d, not same-window), leak added/scored/removed, honest number kept
- [x] 1 limitation named, tied to the guide's panel-imbalance warning